# RAG Retrieval Exploration

This notebook helps inspect how chunk retrieval works in the current project.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config.settings import settings
from src.vectordb.chroma_client import ChromaIndexer

CHROMA_PATH = str((project_root / settings.chroma_path).resolve())
print("Using chroma path:", CHROMA_PATH)

Using chroma path: C:\Users\Andrii\Desktop\rag_system\chroma_db


In [2]:
collection_name = settings.collection_name  #'wiki_simple'

indexer = ChromaIndexer(CHROMA_PATH, collection_name, settings.embedding_model)
try:
    indexer.get_collection()
except Exception as exc:
    raise RuntimeError(
        f"Collection '{collection_name}' not found in {CHROMA_PATH}.\n"
    ) from exc

print('-----------------------------')
print("Collection:", collection_name)
print("Embedding model:", settings.embedding_model)
print("top_k:", settings.top_k)
print("max_distance:", settings.max_distance)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-----------------------------
Collection: wiki_simple
Embedding model: BAAI/bge-large-en-v1.5
top_k: 3
max_distance: 0.75


In [3]:
query = 'Told me about Bernard of Clairvaux'
TOP_K = 5

res = indexer.search(query, top_k=TOP_K)
docs = res['documents'][0]
metas = res['metadatas'][0]
dists = res['distances'][0]

print(f'Query: {query}')
print('-----------------------------')
for i, (doc, meta, dist) in enumerate(zip(docs, metas, dists), start=1):
    title = meta.get('title', 'unknown')
    doc_id = meta.get('doc_id', None)
    chunk_id = meta.get('chunk_id', None)
    preview = doc[:180].replace('\n', ' ')
    print(f'#{i} | dist={dist:.4f} | title={title} | doc_id={doc_id} | chunk_id={chunk_id}')
    print(f'    {preview}...')
    print()

Query: Told me about Bernard of Clairvaux
-----------------------------
#1 | dist=0.5567 | title=Bernard of Clairvaux | doc_id=96039 | chunk_id=0
    Bernard of Clairvaux (1090-1153) was a French theologian. He is a Roman Catholic saint. He was an abbot in the Cistercian order. He was also the patron of the Knights Templar. Rela...

#2 | dist=0.7655 | title=Pope Francis | doc_id=89605 | chunk_id=4
    Francis of Assisi. Just after he was elected, Francis told a newspaper how he chose the new name: "Let me tell you a story," he said. He then [explained] how during the conclave he...

#3 | dist=0.7736 | title=1136 | doc_id=88158 | chunk_id=1
    Peter Abelard writes the Historia Calamitatum, describing his relationship with Heloise. Births Amalric I of Jerusalem William of Newburgh, English historian (d. 1198) Deaths April...

#4 | dist=0.7797 | title=History of the Catholic Church | doc_id=195091 | chunk_id=20
    Cistercian monk Bernard of Clairvaux had great influence over the new ord

In [4]:
MAX_DISTANCE = settings.max_distance
filtered = [(d, m, s) for d, m, s in zip(docs, metas, dists) if s <= MAX_DISTANCE]

print(f'Filtered by max_distance={MAX_DISTANCE}: {len(filtered)} chunks')
for i, (_, meta, dist) in enumerate(filtered, start=1):
    print(f"#{i} | dist={dist:.4f} | title={meta.get('title')} | doc_id={meta.get('doc_id')} | chunk_id={meta.get('chunk_id')}")

Filtered by max_distance=0.75: 1 chunks
#1 | dist=0.5567 | title=Bernard of Clairvaux | doc_id=96039 | chunk_id=0


In [5]:

small_to_big_enabled = settings.small_to_big_enabled
print(f"Small-to-big window enabled: {small_to_big_enabled}")

# Collection the final context (top_k chunks after filtering)
context_parts = []

if small_to_big_enabled:
    WINDOW = settings.small_to_big_window
    print(f'Using window={WINDOW}')
    print('-----------------------------')

    all_window_chunks = []
    for i, (_, meta, dist) in enumerate(filtered[:settings.top_k], start=1):
        doc_id = meta.get('doc_id')
        chunk_id = meta.get('chunk_id')
        title = meta.get('title')

        window_docs = indexer.get_chunk_window(doc_id=doc_id, chunk_id=chunk_id, window=WINDOW)
        print(f'Hit #{i} | title={title} | dist={dist:.4f} | center=({doc_id}, {chunk_id})')
        for j, wdoc in enumerate(window_docs, start=1):
            wpreview = wdoc[:140].replace('\n', ' ')
            print(f'   window chunk {j}: {wpreview}...')
        print()
        all_window_chunks.extend(window_docs)

    # Delete duplicates while preserving order
    unique_chunks = list(dict.fromkeys(all_window_chunks))
    for doc in unique_chunks:
        context_parts.append(doc[:settings.context_chars_per_chunk])
else:
    for doc, meta, dist in filtered:
        context_parts.append(doc[:settings.context_chars_per_chunk])

# Collect the full context
full_context = "\n\n".join(context_parts)

Small-to-big window enabled: False


## How retrieval works

1. Convert the query into an embedding.
2. Search top-k similar chunks in ChromaDB.
3. Filter chunks by `max_distance`.
4. If `small_to_big_enabled=True`, expand a chunk with its neighbors.
5. Join the selected text and pass it to the generator.


In [6]:
# Print gpt2 generation parameters

try:
    from src.models.local.gpt2 import GPT2Generator

    print("GPT-2 generation parameters:", "\n-----------------------------")

    params = [
        "generation_model_name",
        "generation_max_new_tokens",
        "generation_do_sample",
        "generation_temperature",
        "generation_top_p",
        "generation_top_k",
        "generation_max_input_length",
        "generation_no_repeat_ngram_size"
    ]

    for p in params:
        if hasattr(settings, p):
            print(f"{p}: {getattr(settings, p)}")
        else:
            print(f"{p}: -")

except ImportError as e:
    print(e)

GPT-2 generation parameters: 
-----------------------------
generation_model_name: gpt2
generation_max_new_tokens: 40
generation_do_sample: False
generation_temperature: 0.6
generation_top_p: 0.8
generation_top_k: -
generation_max_input_length: 512
generation_no_repeat_ngram_size: None


In [7]:
# Generate the prompt and show it
prompt = settings.prompt_template.format(context=full_context, query=query)
print("=== PROMPT TO GENERATOR ===")
print(prompt)
print("===========================")

# Generate the answer and show it
generator = GPT2Generator()
answer = generator.generate(prompt, max_new_tokens=settings.generation_max_new_tokens)
print("=== GENERATED ANSWER ===")
print(answer)

=== PROMPT TO GENERATOR ===
Context: Bernard of Clairvaux (1090-1153) was a French theologian. He is a Roman Catholic saint. He was an abbot in the Cistercian order. He was also the patron of the Knights Templar. Related pages The road to hell is paved with good intentions Other websites Bernard's entry at Catholic Encyclopedia French Roman Catholics Christian saints French theologians 1090 births 1153 deaths

Question: Told me about Bernard of Clairvaux
Answer:


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

=== GENERATED ANSWER ===
Bernard of Clairvaux was a French theologian. He was a Roman Catholic saint. He was also the patron of the Knights Templar. Related pages Bernard's entry at Catholic Encyclopedia French Roman Catholics Christian
